# Perkerra (P) — After irrigation (AI) — GA-G

This notebook fits the standard gravity-driven Green–Ampt formulation (GA-G)
to after-irrigation samples treated as **non-preferential-flow** observations.
It evaluates: (1) fully optimised parameters, (2) fixed ψ, and
(3) measured Δθ held fixed.

**Input workbook:** `data/Infiltration_P_sat_GA.xlsx`  
**Excluded worksheets:** 1.4, 1.5, 2.2, 2.4, 2.5, 3.4, 3.5, 4.1, 4.2, 4.3


The input data are not distributed with this repository.


In [ ]:
from pathlib import Path
import math
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.metrics import mean_squared_error, r2_score

# The notebooks can be run either from the project root or from /notebooks.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PSI_FIXED_CM = -31.63
INITIAL_PARAMS = np.array([0.0001, 0.35, PSI_FIXED_CM], dtype=float)
PSI_BOUNDS = (-156.5, -4.08)
KFS_BOUNDS = (0.00001, 1.0)

INPUT_FILE = DATA_DIR / "Infiltration_P_sat_GA.xlsx"
OUTPUT_STEM = "P_AI_GA_G"
EXCLUDED_SHEETS = ['1.4', '1.5', '2.2', '2.4', '2.5', '3.4', '3.5', '4.1', '4.2', '4.3']
DELTA_THETA_BOUNDS = (0.12, 0.52)
INITIAL_PARAMS_FIXED_DELTA = np.array([0.0001, PSI_FIXED_CM], dtype=float)
FIXED_DELTA_BOUNDS = [KFS_BOUNDS, PSI_BOUNDS]

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Input workbook not found: {INPUT_FILE}\n"
        "Place the requested workbook in the project data/ directory."
    )


In [ ]:
def green_ampt_time(I, Kfs, delta_theta, psi, H0):
    """Predict elapsed time (min) from cumulative infiltration I (cm)."""
    I = np.asarray(I, dtype=float)
    denominator = delta_theta * (H0 - psi)
    with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
        return (1.0 / (Kfs * (1.0 - delta_theta))) * (
            I
            - (denominator / (1.0 - delta_theta))
            * np.log1p((I * (1.0 - delta_theta)) / denominator)
        )


def objective(params, I_obs, t_obs, H0):
    """Sum of squared errors used for bounded optimisation."""
    Kfs, delta_theta, psi = params
    t_pred = green_ampt_time(I_obs, Kfs, delta_theta, psi, H0)
    if not np.all(np.isfinite(t_pred)):
        return np.inf
    return float(np.sum((t_obs - t_pred) ** 2))


def objective_fixed_delta_theta(params, I_obs, t_obs, H0, delta_theta):
    """Objective when measured delta theta is held fixed."""
    Kfs, psi = params
    t_pred = green_ampt_time(I_obs, Kfs, delta_theta, psi, H0)
    if not np.all(np.isfinite(t_pred)):
        return np.inf
    return float(np.sum((t_obs - t_pred) ** 2))


def rmse(y_obs, y_pred):
    return float(np.sqrt(mean_squared_error(y_obs, y_pred)))


def r_squared(y_obs, y_pred):
    return float(r2_score(y_obs, y_pred))


def kfs_from_last_point(I_last, t_last, delta_theta, psi, H0):
    """Calculate Kfs from the final observation while psi is fixed."""
    denominator = delta_theta * (H0 - psi)
    return (1.0 / (t_last * (1.0 - delta_theta))) * (
        I_last
        - (denominator / (1.0 - delta_theta))
        * np.log1p((I_last * (1.0 - delta_theta)) / denominator)
    )


def load_sheet(xls, sheet_name, required_columns):
    """Load, validate and clean one worksheet."""
    df = pd.read_excel(xls, sheet_name=sheet_name)
    missing = [column for column in required_columns if column not in df.columns]
    if missing:
        raise KeyError(
            f"Worksheet {sheet_name!r} is missing required columns: {missing}"
        )

    clean = df.loc[:, required_columns].copy()
    for column in required_columns:
        clean[column] = pd.to_numeric(clean[column], errors="coerce")
    clean = clean.dropna()

    if len(clean) < 3:
        raise ValueError(
            f"Worksheet {sheet_name!r} has fewer than three complete observations."
        )
    return clean


In [ ]:
xls = pd.ExcelFile(INPUT_FILE)
sheets = [sheet for sheet in xls.sheet_names if sheet not in EXCLUDED_SHEETS]

if not sheets:
    raise ValueError("No worksheets remain after applying the exclusion list.")

required_columns = [
    "Ia",
    "Start Time (min)",
    "Start Level (cm)",
    "Delta Theta",
]
n_cols = 4
n_rows = math.ceil(len(sheets) / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(24, n_rows * 5.5), squeeze=False)
axes = axes.ravel()
result_rows = []

for i, sheet_name in enumerate(sheets):
    try:
        df = load_sheet(xls, sheet_name, required_columns)
        I_obs = df["Ia"].to_numpy(dtype=float)
        t_obs = df["Start Time (min)"].to_numpy(dtype=float)
        H0 = float(df["Start Level (cm)"].iloc[0])
        delta_theta_measured = float(df["Delta Theta"].iloc[0])

        bounds = [KFS_BOUNDS, DELTA_THETA_BOUNDS, PSI_BOUNDS]
        result = minimize(
            objective,
            INITIAL_PARAMS,
            args=(I_obs, t_obs, H0),
            method="L-BFGS-B",
            bounds=bounds,
        )
        result_fixed_delta = minimize(
            objective_fixed_delta_theta,
            INITIAL_PARAMS_FIXED_DELTA,
            args=(I_obs, t_obs, H0, delta_theta_measured),
            method="L-BFGS-B",
            bounds=FIXED_DELTA_BOUNDS,
        )

        if not result.success:
            warnings.warn(
                f"Optimisation failed for {sheet_name}: {result.message}"
            )
            continue
        if not result_fixed_delta.success:
            warnings.warn(
                f"Fixed-Δθ optimisation failed for {sheet_name}: "
                f"{result_fixed_delta.message}"
            )
            continue

        Kfs_opt, delta_theta_opt, psi_opt = result.x
        t_pred_opt = green_ampt_time(
            I_obs, Kfs_opt, delta_theta_opt, psi_opt, H0
        )
        rmse_opt = rmse(t_obs, t_pred_opt)
        r2_opt = r_squared(t_obs, t_pred_opt)

        Kfs_fixed_psi = kfs_from_last_point(
            I_obs[-1], t_obs[-1], delta_theta_opt, PSI_FIXED_CM, H0
        )
        t_pred_fixed_psi = green_ampt_time(
            I_obs, Kfs_fixed_psi, delta_theta_opt, PSI_FIXED_CM, H0
        )
        rmse_fixed_psi = rmse(t_obs, t_pred_fixed_psi)
        r2_fixed_psi = r_squared(t_obs, t_pred_fixed_psi)

        Kfs_fixed_delta, psi_fixed_delta = result_fixed_delta.x
        t_pred_fixed_delta = green_ampt_time(
            I_obs,
            Kfs_fixed_delta,
            delta_theta_measured,
            psi_fixed_delta,
            H0,
        )
        rmse_fixed_delta = rmse(t_obs, t_pred_fixed_delta)
        r2_fixed_delta = r_squared(t_obs, t_pred_fixed_delta)

        result_rows.extend(
            [
                {
                    "Sample": sheet_name,
                    "Model": "GA-G",
                    "Parameterisation": "Optimised",
                    "Psi_cm": psi_opt,
                    "Delta_theta": delta_theta_opt,
                    "Kfs_cm_min": Kfs_opt,
                    "RMSE_min": rmse_opt,
                    "R2": r2_opt,
                },
                {
                    "Sample": sheet_name,
                    "Model": "GA-G",
                    "Parameterisation": "Fixed psi",
                    "Psi_cm": PSI_FIXED_CM,
                    "Delta_theta": delta_theta_opt,
                    "Kfs_cm_min": Kfs_fixed_psi,
                    "RMSE_min": rmse_fixed_psi,
                    "R2": r2_fixed_psi,
                },
                {
                    "Sample": sheet_name,
                    "Model": "GA-G",
                    "Parameterisation": "Fixed measured delta theta",
                    "Psi_cm": psi_fixed_delta,
                    "Delta_theta": delta_theta_measured,
                    "Kfs_cm_min": Kfs_fixed_delta,
                    "RMSE_min": rmse_fixed_delta,
                    "R2": r2_fixed_delta,
                },
            ]
        )

        ax = axes[i]
        ax.scatter(t_obs, I_obs, color="black", label="Observed data")
        ax.plot(t_pred_opt, I_obs, color="blue", linestyle="-", label="Optimised")
        ax.plot(
            t_pred_fixed_psi,
            I_obs,
            color="red",
            linestyle="--",
            label="Fixed ψ",
        )
        ax.plot(
            t_pred_fixed_delta,
            I_obs,
            color="darkgreen",
            linestyle=":",
            label="Fixed Δθ",
        )

        optimal_text = "\n".join(
            [
                "Optimised",
                f"ψ = {psi_opt:.2f} cm",
                f"Δθ = {delta_theta_opt:.2f}",
                f"$K_{{fs}}$ = {Kfs_opt:.4f} cm/min",
                f"RMSE = {rmse_opt:.2f} min",
                f"$R^2$ = {r2_opt:.2f}",
            ]
        )
        fixed_psi_text = "\n".join(
            [
                f"Fixed ψ = {PSI_FIXED_CM:.2f} cm",
                f"Δθ = {delta_theta_opt:.2f}",
                f"$K_{{fs}}$ = {Kfs_fixed_psi:.4f} cm/min",
                f"RMSE = {rmse_fixed_psi:.2f} min",
                f"$R^2$ = {r2_fixed_psi:.2f}",
            ]
        )
        fixed_delta_text = "\n".join(
            [
                "Fixed measured Δθ",
                f"ψ = {psi_fixed_delta:.2f} cm",
                f"Δθ = {delta_theta_measured:.2f}",
                f"$K_{{fs}}$ = {Kfs_fixed_delta:.4f} cm/min",
                f"RMSE = {rmse_fixed_delta:.2f} min",
                f"$R^2$ = {r2_fixed_delta:.2f}",
            ]
        )

        ax.text(
            0.04,
            0.96,
            optimal_text,
            transform=ax.transAxes,
            va="top",
            fontsize=9,
            bbox={"boxstyle": "round,pad=0.3", "facecolor": "none"},
        )
        ax.text(
            0.96,
            0.04,
            fixed_psi_text,
            transform=ax.transAxes,
            va="bottom",
            ha="right",
            fontsize=9,
            bbox={"boxstyle": "round,pad=0.3", "facecolor": "none"},
        )
        ax.text(
            0.96,
            0.52,
            fixed_delta_text,
            transform=ax.transAxes,
            va="center",
            ha="right",
            fontsize=9,
            bbox={"boxstyle": "round,pad=0.3", "facecolor": "none"},
        )
        ax.set_title(sheet_name)
        ax.set_xlabel("t (min)")
        ax.set_ylabel("I (cm)")
        ax.grid(True)

    except Exception as exc:
        warnings.warn(f"Worksheet {sheet_name!r} was skipped: {exc}")

for ax in axes[len(sheets):]:
    ax.remove()

handles = [
    plt.Line2D([0], [0], marker="o", color="black", linestyle="None"),
    plt.Line2D([0], [0], color="blue", linestyle="-"),
    plt.Line2D([0], [0], color="red", linestyle="--"),
    plt.Line2D([0], [0], color="darkgreen", linestyle=":"),
]
labels = [
    "Observed data",
    "Optimised GA-G",
    f"GA-G with ψ = {PSI_FIXED_CM:.2f} cm",
    "GA-G with measured Δθ",
]
fig.legend(handles, labels, loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.01))
fig.tight_layout(rect=(0, 0.05, 1, 1))

plot_path = OUTPUT_DIR / f"{OUTPUT_STEM}_plots.png"
results_path = OUTPUT_DIR / f"{OUTPUT_STEM}_results.csv"
fig.savefig(plot_path, dpi=300, bbox_inches="tight")
plt.show()

results_df = pd.DataFrame(result_rows)
results_df.to_csv(results_path, index=False)

print(f"Processed worksheets: {results_df['Sample'].nunique() if not results_df.empty else 0}")
print(f"Results: {results_path}")
print(f"Figure:  {plot_path}")
results_df
